In [1]:
import os
import yaml
import torch
import argparse
import numpy as np
import pandas as pd
from tqdm.autonotebook import trange
from datetime import datetime
from types import SimpleNamespace
from torchmetrics import AUROC, Accuracy
import torch_geometric as pyg
from task_constructor import UnifiedTaskConstructor

import utils
from utils import (
    SentenceEncoder,
    MultiApr,
    MultiAuc,
)

from gp.lightning.module_template import ExpConfig
from gp.lightning.training import lightning_fit
from gp.lightning.data_template import DataModule
from gp.utils.utils import (
    load_yaml,
    combine_dict,
    merge_mod,
    setup_exp,
    set_random_seed,
)
from gp.lightning.metric import (
    flat_binary_func,
    EvalKit,
)

from models.model import PyGRGCNEdge
from models.model import BinGraphModel, BinGraphAttModel


from lightning_model import GraphPredLightning
from pytorch_lightning.loggers import WandbLogger

from torch_geometric.data import InMemoryDataset
from torch_geometric.datasets import Planetoid

ModuleNotFoundError: No module named 'task_constructor'

ModuleNotFoundError: No module named 'pyg'

In [2]:
import torch
print(torch.__version__)                # 查看pytorch安装的版本号
print(torch.cuda.is_available())        # 查看cuda是否可用。True为可用，即是gpu版本pytorch
print(torch.cuda.get_device_name(0))    # 返回GPU型号
print(torch.cuda.device_count())        # 返回可以用的cuda（GPU）数量，0代表一个
print(torch.version.cuda)     

1.13.1+cu116
True
NVIDIA GeForce RTX 4070
1
11.6


In [4]:
parser = argparse.ArgumentParser(description="rl")
parser.add_argument("--override", type=str)

parser.add_argument(
        "opts",
        default=[],
        nargs=argparse.REMAINDER,
        help="Modify config options using the command-line",
    )


_StoreAction(option_strings=[], dest='opts', nargs='...', const=None, default=[], type=None, choices=None, required=True, help='Modify config options using the command-line', metavar=None)

In [4]:
# params = parser.parse_args()
# Namespace(override='e2e_all_config.yaml', opts=[])
# Namespace(override='e2e_all_config.yaml', opts=['num_layers', '7', 'batch_size', '512', 'dropout', '0.15', 'JK', 'none'])

In [3]:
configs = []
configs.append(
    load_yaml(
        os.path.join(
            #os.path.dirname(__file__), "configs", "default_config.yaml"
            r'G:\Research\GRAPH_TORCH_TEST', "configs", "default_config.yaml"
        )
    )
)
#configs

In [4]:
#if params.override is not None:
# 添加新的config
override_config = load_yaml('e2e_all_config_omit.yaml')
configs.append(override_config)

In [5]:
mod_params = combine_dict(*configs)
#mod_params = merge_mod(mod_params, params.opts)
mod_params = merge_mod(mod_params, [])

#setup_exp(mod_params)
curtime = datetime.now()
exp_name = str(curtime).replace(" ", "_")
exp_name = exp_name.replace(":", "_")
exp_name = exp_name.replace(".", "_")
exp_name = exp_name.split("_")[0]
exp_dir = os.path.join("./saved_exp", exp_name)
if not os.path.exists(exp_dir):
    os.makedirs(exp_dir)
with open(os.path.join(exp_dir, "command"), "w") as f:
    yaml.dump(mod_params, f)
mod_params["exp_dir"] = exp_dir
params = SimpleNamespace(**mod_params)
set_random_seed(params.seed)
torch.set_float32_matmul_precision("high")
params.log_project = "full_cdm"

params.exp_name += f"_{params.llm_name}_ofa1"

In [7]:
params

namespace(num_bases=4,
          emb_dim=768,
          num_layers=7,
          dropout=0.15,
          JK='none',
          lr=0.001,
          l2=0,
          num_epochs=50,
          batch_size=512,
          num_workers=4,
          seed=1,
          data_path=None,
          offline_log=True,
          exp_name='_ST_ofa1',
          train_sample_size=-1,
          eval_sample_size=-1,
          rwpe=None,
          task_names=['cora_node'],
          llm_name='ST',
          llm_b_size=1,
          load_texts=False,
          max_nodes_per_hop=100,
          load_best=False,
          test_rep=1,
          val_interval=1,
          eval_batch_size=256,
          d_multiple=[1.5],
          d_min_ratio=[1],
          exp_dir='./saved_exp\\2024-05-06',
          log_project='full_cdm')

草稿

In [8]:
# 读取源文件中的pt文件
cur_path = r'G:\Research\Graph_torch_test\data\single_graph\Cora'
path = os.path.join(cur_path, "cora.pt")
data = torch.load(path)
print(data.keys())

['category_names', 'test_masks', 'val_masks', 'y', 'raw_text', 'edge_index', 'x', 'label_names', 'train_masks', 'raw_texts']


In [9]:
type(data)

torch_geometric.data.data.Data

In [16]:
text = data.raw_texts
label_names = data.label_names  # 分类的标签

In [17]:
nx_g = pyg.utils.to_networkx(data, to_undirected=True)
# 而在实验的过程中，可能需要使用到 networkx 提供的一些功能来实现与图相关的操作，这时图数据需要在两个框架提供的图结构之间进行转换

In [21]:
edge_index = torch.tensor(list(nx_g.edges())).T
# print(edge_index.size())
# 邻接矩阵

In [32]:
data_dict = data.to_dict()  # 转为字典
data_dict["edge_index"] = edge_index   
new_data = pyg.data.data.Data(**data_dict)  # 生成pyg格式，这里还是需要看一下的

In [54]:
category_desc = pd.read_csv(
        os.path.join(r'G:\Research\Graph_torch_test\data\single_graph\Cora', "categories.csv"), 
                     sep=","
    ).values
ordered_desc = []
for i, label in enumerate(label_names):
    true_ind = label == category_desc[:, 0]
    ordered_desc.append((label, category_desc[:, 1][true_ind]))
# 这是用来描述每一个标签的文本信息

In [60]:
def get_logic_label(ordered_txt):
    or_labeled_text = []
    not_and_labeled_text = []
    for i in range(len(ordered_txt)):
        for j in range(len(ordered_txt)):
            c1 = ordered_txt[i]
            c2 = ordered_txt[j]
            txt = "prompt node. literature category and description: not " + c1[0] + ". " + c1[1][0] + " and not " + c2[
                0] + ". " + c2[1][0]
            not_and_labeled_text.append(txt)
            txt = "prompt node. literature category and description: either " + c1[0] + ". " + c1[1][0] + " or " + c2[
                0] + ". " + c2[1][0]
            or_labeled_text.append(txt)
    return or_labeled_text + not_and_labeled_text

In [61]:
clean_text = ["feature node. paper title and abstract: " + t for t in text]   
# Input graph node，处理每一个节点的特征
label_text = [
    "prompt node. literature category and description: "
    + desc[0]
    + "."
    + desc[1][0]
    for desc in ordered_desc
]  
# Node classification task - Class node，对节点分类标签的信息

edge_label_text = [
    "prompt node. two papers do not have co-citation",
    "prompt node. two papers have co-citation"
]
# Link classification task - Class node，对边分类标签的信息

logic_label_text = get_logic_label(ordered_desc)
# or 和 not 的逻辑判断

In [80]:
edge_text = [
    "feature edge. connected papers are cited together by other papers."
]
# Input graph edge

noi_node_edge_text = [
    "prompt node. link prediction on the papers that are cited together"
]
# Prompt node - edge

noi_node_text = [
    "prompt node. node classification on the paper's category"
]
# Prompt node - node

prompt_edge_text = ["prompt edge", "prompt edge. edge for query graph that is our target",
                    "prompt edge. edge for support graph that is an example"]
# Few-shot与Zero-shot的任务描述

In [104]:
torch.arange(len(label_text))

tensor([0, 1, 2, 3, 4, 5, 6])

In [127]:
data_list, texts, side_data = (
        [new_data],
        [
            clean_text,
            edge_text,
            noi_node_text + noi_node_edge_text,
            label_text + edge_label_text + logic_label_text,
            prompt_edge_text,
        ],
        {"e2e_node": {"noi_node_text_feat": ["noi_node_text_feat", [0]],
                      "class_node_text_feat": ["class_node_text_feat", torch.arange(len(label_text))],
                      "prompt_edge_text_feat": ["prompt_edge_text_feat", [0]]},
         "e2e_link": {"noi_node_text_feat": ["noi_node_text_feat", [1]],
                      "class_node_text_feat": ["class_node_text_feat",
                                               torch.arange(len(label_text), len(label_text) + len(edge_label_text))],
                      "prompt_edge_text_feat": ["prompt_edge_text_feat", [0]]},
         "lr_node": {"noi_node_text_feat": ["noi_node_text_feat", [0]],
                     "class_node_text_feat": ["class_node_text_feat", torch.arange(len(label_text))],
                     "prompt_edge_text_feat": ["prompt_edge_text_feat", [0, 1, 2]]},
         "logic_e2e": {"noi_node_text_feat": ["noi_node_text_feat", [0]],
                       "class_node_text_feat": ["class_node_text_feat",
                                                torch.arange(len(label_text) + len(edge_label_text),
                                                             len(label_text) + len(edge_label_text) + len(
                                                                 logic_label_text))],
                       "prompt_edge_text_feat": ["prompt_edge_text_feat", [0]]},
         }
    )

# noi_node_text_feat -- 任务描述
# class_node_text_feat -- 用于分类的标签
# prompt_edge_text_feat -- 暂时不明

In [145]:
def add_raw_texts(self, data_list, texts):
    data_list[0].node_text_feat = np.array(texts[0])
    data_list[0].edge_text_feat = np.array(texts[1])
    data_list[0].noi_node_text_feat = np.array(texts[2])
    data_list[0].class_node_text_feat = np.array(texts[3])
    data_list[0].prompt_edge_text_feat = np.array(texts[4])
    return self.collate(data_list)
def text2feature(texts,encoder):
    if isinstance(texts[0], str):
        return data2vec(encoder,texts)
    return [text2feature(t,encoder) for t in texts]
def data2vec(encoder, data: list[str]) -> torch.Tensor:
    r"""
    Encode a list of string to a len(data)-by-d matrix, where d is the output dimension of the LLM.
    """
    if encoder is None:
        raise NotImplementedError("LLM encoder is not defined")
    if data is None:
        return None
    embeddings = encoder.encode(data).cpu().numpy()
    return embeddings

In [172]:
with torch.no_grad():
    for start_index in trange(0, len(texts[0]), 512, desc="Batches", disable=False, ):
        print(start_index)
        sentences_batch = texts[0][start_index: start_index + 512]
        text_tokens = encoder.model.tokenizer(sentences_batch, return_tensors="pt", padding="longest", truncation=True,
                                    max_length=500).to('cuda')
        embeddings, _ = encoder.model.encode(text_tokens, pooling=True)
        break

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

0


Batches:   0%|          | 0/6 [00:06<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.72 GiB (GPU 0; 11.99 GiB total capacity; 8.91 GiB already allocated; 1.19 GiB free; 8.92 GiB reserved in total by PyTorch) If reserved memory is >> allocated memory try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [171]:
text_tokens.keys()

dict_keys(['input_ids', 'attention_mask'])

In [146]:
texts_emb = text2feature(texts,encoder)
# 这里的操作： tokenizer --> model

Batches: 100%|██████████| 3/3 [00:00<00:00, 214.06it/s]


In [155]:
texts_emb[0][0]

array([ 0.01777354, -0.01039051, -0.04736627, -0.00784677,  0.06084023,
        0.01598853,  0.00416745, -0.00935482,  0.02318387, -0.00575685,
       -0.010334  ,  0.01270784,  0.01525205, -0.05451107, -0.02196415,
       -0.00609458, -0.00520146, -0.05090937,  0.03622211,  0.01750773,
       -0.02402773,  0.03219585, -0.01516439,  0.04299492,  0.01013391,
        0.00567586, -0.00644491,  0.00611462, -0.02879669,  0.04513787,
       -0.05157039,  0.02702896,  0.05001691,  0.0093117 , -0.00286893,
       -0.00283535, -0.05780992, -0.01474014, -0.04366378,  0.002716  ,
       -0.00447445,  0.03890738,  0.02228159,  0.04902431, -0.09425317,
       -0.02099369, -0.10551293, -0.00934653,  0.00776365, -0.02647158,
       -0.01179662,  0.01168877, -0.01410159, -0.03303957, -0.0101285 ,
       -0.03219109,  0.03515811,  0.00415223,  0.0282803 , -0.0217335 ,
       -0.03707037,  0.01856106, -0.01245904,  0.03329601,  0.07067668,
        0.0302432 , -0.02501874, -0.06054574, -0.01468254,  0.02

In [36]:
dataset_origin = Planetoid(root='/Research/GRAPH_TORCH_TEST/Dataset', name='Cora')

In [47]:
dataset_origin.__dict__
# 直接走PQW下来的是经过预处理的，不完整

{'name': 'Cora',
 'split': 'public',
 'root': '\\Research\\GRAPH_TORCH_TEST\\Dataset',
 'transform': None,
 'pre_transform': None,
 'pre_filter': None,
 'log': True,
 '_indices': None,
 'force_reload': False,
 '_data': Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708]),
 'slices': None,
 '_data_list': None}

## 正常的流程

In [6]:
# 0. Check GPU setting.
device, gpu_ids = utils.get_available_devices()
gpu_size = len(gpu_ids)

1. Initiate task constructor.

In [9]:
encoder = utils.SentenceEncoder(params.llm_name, batch_size=params.llm_b_size)

In [10]:
task_config_lookup = load_yaml(
    os.path.join(
        #os.path.dirname(__file__), "configs", "task_config.yaml"
        r'G:\Research\OneForAll', "configs", "task_config.yaml"
        )
)
data_config_lookup = load_yaml(
    os.path.join(
        #os.path.dirname(__file__), "configs", "data_config.yaml"
        r'G:\Research\OneForAll',"configs", "data_config.yaml"
        )
    )
if isinstance(params.task_names, str):
    task_names = [a.strip() for a in params.task_names.split(",")]
else:
    task_names = params.task_names

tasks = UnifiedTaskConstructor(
    task_names,
    params.load_texts,
    encoder,
    task_config_lookup,
    data_config_lookup,
    batch_size=params.batch_size,
    sample_size=params.train_sample_size,
)


In [11]:
val_task_index_lst, val_pool_mode = tasks.construct_exp()

Processing...


torch.Size([2, 5278])


Traceback (most recent call last):
  File "C:\Users\ljh\AppData\Roaming\Python\Python310\site-packages\debugpy\_vendored\pydevd\_pydevd_bundle\pydevd_vars.py", line 624, in change_attr_expression
    value = eval(expression, frame.f_globals, frame.f_locals)
  File "<string>", line 1, in <module>
NameError: name 'array' is not defined
Batches: 100%|██████████| 3/3 [00:00<00:00, 189.11it/s]


Saving...


Done!
C:\Users\ljh\AppData\Roaming\Python\Python310\site-packages\torch_geometric\data\in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
C:\Users\ljh\AppData\Roaming\Python\Python310\site-packages\torch_geometric\data\in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)
C:\Users\ljh\AppData\Roami

基本配置编译完成

核心main函数

In [10]:
# 0. Check GPU setting.
device, gpu_ids = utils.get_available_devices()
gpu_size = len(gpu_ids)

1. Initiate task constructor.

In [11]:
encoder = utils.SentenceEncoder(params.llm_name, batch_size=params.llm_b_size)
task_config_lookup = load_yaml(
    os.path.join(
        #os.path.dirname(__file__), "configs", "task_config.yaml"
        r'G:\Research\OneForAll', "configs", "task_config.yaml"
        )
)
data_config_lookup = load_yaml(
    os.path.join(
        #os.path.dirname(__file__), "configs", "data_config.yaml"
        r'G:\Research\OneForAll',"configs", "data_config.yaml"
        )
    )
if isinstance(params.task_names, str):
    task_names = [a.strip() for a in params.task_names.split(",")]
else:
    task_names = params.task_names

tasks = UnifiedTaskConstructor(
    task_names,
    params.load_texts,
    encoder,
    task_config_lookup,
    data_config_lookup,
    batch_size=params.batch_size,
    sample_size=params.train_sample_size,
)


In [34]:
print(tasks.task_config_lookup['cora_node'])  # 这个就是stage-config
print(tasks.data_config_lookup['cora_node']) # 这个就是data-config

{'eval_pool_mode': 'mean', 'eval_set_constructs': [{'stage': 'train', 'split_name': 'train'}, {'stage': 'valid', 'split_name': 'valid'}, {'stage': 'test', 'split_name': 'test'}, {'stage': 'test', 'split_name': 'train'}], 'dataset': 'cora_node'}
{'task_level': 'e2e_node', 'preprocess': None, 'construct': 'ConstructNodeCls', 'args': {'walk_length': None, 'single_prompt_edge': False, 'max_nodes_per_hop': 100}, 'eval_metric': 'acc', 'eval_func': 'classification_func', 'eval_mode': 'max', 'dataset_name': 'Cora', 'dataset_splitter': 'CiteSplitter', 'process_label_func': 'process_int_label', 'num_classes': 7}


In [35]:
val_task_index_lst, val_pool_mode = tasks.construct_exp()

C:\Users\ljh\AppData\Roaming\Python\Python310\site-packages\torch_geometric\data\in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


In [173]:
# remove llm model
if encoder is not None:
    encoder.flush_model()
# encoder.model

2.Load model 

In [14]:
out_dim = params.emb_dim + (params.rwpe if params.rwpe is not None else 0)

gnn = PyGRGCNEdge(
    params.num_layers,
    5,
    out_dim,
    out_dim,
    drop_ratio=params.dropout,
    JK=params.JK,
)

bin_model = BinGraphAttModel if params.JK == "none" else BinGraphModel
model = bin_model(model=gnn, llm_name=params.llm_name, outdim=out_dim, task_dim=1,
                    add_rwpe=params.rwpe, dropout=params.dropout)

In [15]:
model

BinGraphAttModel(
  (model): PyGRGCNEdge(
    (conv): ModuleList(
      (0): RGCNEdgeConv(768, 768)
      (1): RGCNEdgeConv(768, 768)
      (2): RGCNEdgeConv(768, 768)
      (3): RGCNEdgeConv(768, 768)
      (4): RGCNEdgeConv(768, 768)
      (5): RGCNEdgeConv(768, 768)
      (6): RGCNEdgeConv(768, 768)
    )
    (batch_norm): ModuleList(
      (0): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (1): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (3): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (4): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
  )
  (llm

3. Construct datasets and lightning datamodule.

In [15]:
if hasattr(params, "d_multiple"):
    if isinstance(params.d_multiple, str):
        data_multiple = [float(a) for a in params.d_multiple.split(",")]
    else:
        data_multiple = params.d_multiple
else:
    data_multiple = [1]

if hasattr(params, "d_min_ratio"):
    if isinstance(params.d_min_ratio, str):
        min_ratio = [float(a) for a in params.d_min_ratio.split(",")]
    else:
        min_ratio = params.d_min_ratio
else:
    min_ratio = [1]

In [16]:
train_data = tasks.make_train_data(data_multiple, min_ratio, data_val_index=val_task_index_lst)
text_dataset = tasks.make_full_dm_list(
    data_multiple, min_ratio, train_data
)
params.datamodule = DataModule(
    text_dataset, gpu_size=gpu_size, num_workers=params.num_workers
)


4. Initiate evaluation kit. 

In [17]:
eval_data = text_dataset["val"] + text_dataset["test"]
val_state = [dt.state_name for dt in text_dataset["val"]]
test_state = [dt.state_name for dt in text_dataset["test"]]
eval_state = val_state + test_state
eval_metric = [dt.metric for dt in eval_data]
eval_funcs = [dt.meta_data["eval_func"] for dt in eval_data]
loss = torch.nn.BCEWithLogitsLoss()
evlter = []

In [18]:
for dt in eval_data:
    if dt.metric == "acc":
        evlter.append(Accuracy(task="multiclass", num_classes=dt.classes))
    elif dt.metric == "auc":
        evlter.append(AUROC(task="binary"))
    elif dt.metric == "apr":
        evlter.append(MultiApr(num_labels=dt.classes))
    elif dt.metric == "aucmulti":
        evlter.append(MultiAuc(num_labels=dt.classes))

In [19]:
metrics = EvalKit(
    eval_metric,
    evlter,
    loss,
    eval_funcs,
    flat_binary_func,
    eval_mode="max",
    exp_prefix="",
    eval_state=eval_state,
    val_monitor_state=val_state[0],
    test_monitor_state=test_state[0],
)

5. Initiate optimizer, scheduler and lightning model module.

In [20]:
optimizer = torch.optim.Adam(
    model.parameters(), lr=params.lr, weight_decay=params.l2
)
lr_scheduler = {
    "scheduler": torch.optim.lr_scheduler.StepLR(optimizer, 15, 0.5),
    "interval": "epoch",
    "frequency": 1,
}

In [21]:
exp_config = ExpConfig(
    "",
    optimizer,
    dataset_callback=train_data.update,
    lr_scheduler=lr_scheduler,
)
exp_config.val_state_name = val_state
exp_config.test_state_name = test_state

In [22]:
pred_model = GraphPredLightning(exp_config, model, metrics)

6. Start training and logging.

In [23]:
wandb_logger = WandbLogger(
    project=params.log_project,
    name=params.exp_name,
    save_dir=params.exp_dir,
    offline=params.offline_log,
)


In [24]:


strategy = "deepspeed_stage_2" if gpu_size > 1 else "auto"
val_res, test_res = lightning_fit(
    wandb_logger,
    pred_model,
    params.datamodule,
    metrics,
    params.num_epochs,
    strategy=strategy,
    save_model=False,
    load_best=params.load_best,
    reload_freq=1,
    test_rep=params.test_rep,
    val_interval=params.val_interval
)



GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id 27288zdg.


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name     | Type             | Params
----------------------------------------------
0 | model    | BinGraphAttModel | 28.9 M
1 | eval_kit | EvalKit          | 0     
----------------------------------------------
28.9 M    Trainable params
0         Non-trainable params
28.9 M    Total params
115.704   Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

C:\Users\ljh\AppData\Roaming\Python\Python310\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:436: Consider setting `persistent_workers=True` in 'val_dataloader' to speed up the dataloader worker initialization.


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:02<00:00,  0.69it/s]

C:\Users\ljh\AppData\Roaming\Python\Python310\site-packages\torchmetrics\utilities\prints.py:43: UserWarning: No positive samples in targets, true positive value should be meaningless. Returning zero tensor in true positive score
  warnings.warn(*args, **kwargs)  # noqa: B028


In [15]:
val_pool_mode

['mean', 'mean']